# Capital IQ: ISIN / CUSIP / Ticker <-> Capital IQ Company ID

从本地 Capital IQ helper 数据（`Z:/dataset/Capital_IQ/helper/`）提取三类映射：

| 输出文件 | 源表 | 映射关系 |
|---|---|---|
| `isin/ciq_company_id.pq` | `wrds_isin` | ISIN <-> ciq_company_id |
| `isin/ciq_company_id-cusip.pq` | `wrds_cusip` | ciq_company_id <-> CUSIP |
| `isin/ciq_company_id-ticker.pq` | `wrds_ticker` | ciq_company_id <-> Ticker |

所有映射的 `source` 为 `Capital IQ`，`update` 取数据文件的 mtime。

**增量更新**：与已有 `.pq` 文件合并——保留旧记录（即使新数据中已不存在），新增记录，对已有记录更新 `update` 为最新时间。

**修复记录**：`ciq_company_id-cusip.pq` 之前误存为 ISIN <-> company_name 映射（与 `company_name.pq` 内容重复），本笔记本修正为正确的 ciq_company_id <-> CUSIP 映射。

In [1]:
import pandas as pd
import os
from datetime import datetime

# 路径配置（按实际环境修改）
CIQ_DATA_DIR = 'Z:/dataset/Capital_IQ/helper'
OUTPUT_DIR = '../isin'

## 0. 检查数据路径

列出 `CIQ_DATA_DIR` 下的文件，确认源数据存在。

In [2]:
if os.path.exists(CIQ_DATA_DIR):
    for f in sorted(os.listdir(CIQ_DATA_DIR)):
        p = os.path.join(CIQ_DATA_DIR, f)
        tag = 'DIR' if os.path.isdir(p) else 'FILE'
        print(f'  [{tag}] {f}')
else:
    print(f'[error] 路径不存在: {CIQ_DATA_DIR}')
    print('请修改 CIQ_DATA_DIR 后重试')

  [DIR] 2025upd
  [DIR] chars
  [DIR] ciqaddress
  [DIR] ciqaddresstype
  [DIR] ciqbusinessdescription
  [DIR] ciqbusinessdescriptionlong
  [DIR] ciqchartype
  [DIR] ciqchartypetosubtype
  [DIR] ciqcommittee
  [DIR] ciqcompany
  [DIR] ciqcompanyindustrytree
  [DIR] ciqcompanystatustype
  [DIR] ciqcompanytype
  [DIR] ciqconstituent
  [DIR] ciqcountrygeo
  [DIR] ciqcurrency
  [DIR] ciqexchange
  [DIR] ciqexchangerate
  [DIR] ciqgvkeyiid
  [DIR] ciqindex
  [DIR] ciqindexconstituent
  [DIR] ciqindexprovider
  [DIR] ciqindexrel
  [DIR] ciqindextradingitem
  [DIR] ciqindextradingitemtype
  [DIR] ciqindexvalue
  [DIR] ciqindustrytosic
  [DIR] ciqobjecttype
  [DIR] ciqperiodtype
  [DIR] ciqreportingtemplatetype
  [DIR] ciqrestatementtype
  [DIR] ciqsecurity
  [DIR] ciqsecuritygroup
  [DIR] ciqsecuritysubtype
  [DIR] ciqsecuritytype
  [DIR] ciqstate
  [DIR] ciqsubtype
  [DIR] ciqsymbol
  [DIR] ciqsymboltype
  [DIR] ciqtradingitem
  [DIR] ciqtradingitemstatus
  [DIR] ciqvaluetype
  [DIR] wrds_ci

## 1. 辅助函数

In [3]:
def find_data_path(table_name):
    """查找 wrds 表对应的 parquet 文件或目录"""
    for ext in ['', '.pq', '.parquet']:
        path = os.path.join(CIQ_DATA_DIR, f'{table_name}{ext}')
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f'找不到 {table_name}，请检查 CIQ_DATA_DIR={CIQ_DATA_DIR}')


def get_data_mtime(table_name):
    """获取数据文件的修改时间作为 update 日期"""
    path = find_data_path(table_name)
    if os.path.isdir(path):
        # 分区 parquet 目录：取第一个 part 文件的 mtime
        parts = sorted(f for f in os.listdir(path)
                        if f.endswith(('.pq', '.parquet')))
        if parts:
            path = os.path.join(path, parts[0])
    return pd.to_datetime(datetime.fromtimestamp(os.path.getmtime(path)).date())


def process_ciq(table_name, value_col, isin_centered=False):
    """读取 ciq helper 表（parquet），清洗并返回 DataFrame。

    Parameters
    ----------
    table_name    : str  源表名，如 'wrds_isin'
    value_col     : str  值列名，如 'isin' / 'cusip' / 'ticker'
    isin_centered : bool True 时输出 [value_col, ciq_company_id]；
                          False 时输出 [ciq_company_id, value_col]
    """
    data_path = find_data_path(table_name)
    print(f'-- {table_name} <- {data_path} --')

    df = pd.read_parquet(data_path)
    df.columns = df.columns.str.lower()
    print(f'  原始: {df.shape}, cols={list(df.columns)}')

    df = df[['companyid', value_col]].copy()
    df.rename(columns={'companyid': 'ciq_company_id'}, inplace=True)
    df.dropna(subset=['ciq_company_id', value_col], inplace=True)

    # 类型转换
    df['ciq_company_id'] = df['ciq_company_id'].map(int)
    if pd.api.types.is_numeric_dtype(df[value_col]):
        df[value_col] = df[value_col].map(int).astype(str)
    else:
        df[value_col] = df[value_col].astype(str)
    df[value_col] = df[value_col].str.strip().str.upper()
    df = df[df[value_col] != '']

    df.drop_duplicates(subset=['ciq_company_id', value_col], inplace=True)
    print(f'  清洗后: {df.shape}')

    df['source'] = 'Capital IQ'
    df['update'] = get_data_mtime(table_name)

    cols = ([value_col, 'ciq_company_id'] if isin_centered
            else ['ciq_company_id', value_col]) + ['source', 'update']
    df = df[cols].sort_values(cols[:2]).reset_index(drop=True)
    return df


def incremental_update(new_df, output_path, key_cols):
    """将新数据与已有数据增量合并。

    - 已有文件不存在 -> 首次创建
    - schema 不匹配（如旧文件数据有误）-> 直接覆盖
    - 正常合并: old + new concat，按 update 排序后去重保留最新
    """
    if not os.path.exists(output_path):
        print(f'  首次创建: {len(new_df):,} rows')
        new_df.to_parquet(output_path, index=False)
        return new_df

    old = pd.read_parquet(output_path)
    if '__index_level_0__' in old.columns:
        old = old.drop(columns=['__index_level_0__'])

    if list(old.columns) != list(new_df.columns):
        print(f'  [warn] schema 不匹配，直接覆盖')
        print(f'    old: {list(old.columns)}')
        print(f'    new: {list(new_df.columns)}')
        new_df.to_parquet(output_path, index=False)
        return new_df

    n_old = len(old)
    combined = pd.concat([old, new_df], ignore_index=True)
    combined = combined.sort_values('update')
    combined = combined.drop_duplicates(subset=key_cols, keep='last')
    combined = combined.sort_values(key_cols).reset_index(drop=True)

    n_added = len(combined) - n_old
    print(f'  增量合并: old {n_old:,} + new {len(new_df):,} -> {len(combined):,} (净增 {n_added:+,})')
    combined.to_parquet(output_path, index=False)
    return combined

## 2. ISIN <-> Capital IQ Company ID

源表 `wrds_isin`（字段：companyid, isin, companyname）。输出 `ciq_company_id.pq`。

In [4]:
df_isin = process_ciq('wrds_isin', 'isin', isin_centered=True)
df_isin = incremental_update(df_isin, f'{OUTPUT_DIR}/ciq_company_id.pq', ['isin', 'ciq_company_id'])
df_isin.head(3)

-- wrds_isin <- Z:/dataset/Capital_IQ/helper\wrds_isin --
  原始: (17749928, 6), cols=['companyid', 'isin', 'companyname', 'startdate', 'enddate', 'primaryflag']
  清洗后: (17748909, 2)
  增量合并: old 9,600,285 + new 17,748,909 -> 18,034,595 (净增 +8,434,310)


,isin,ciq_company_id,source,update
0,AD000A0MYR70,216404615,Capital IQ,2026-03-26
1,AD000A1J3BL7,288620627,Capital IQ,2026-03-26
2,AD000A1KBUP7,224075352,Capital IQ,2026-03-26


## 3. Capital IQ Company ID <-> CUSIP

源表 `wrds_cusip`（字段：companyid, cusip, companyname）。输出 `ciq_company_id-cusip.pq`。

此表修复了之前的数据错误（旧文件内容为 ISIN <-> company_name，与 `company_name.pq` 重复）。
由于旧文件 schema 不匹配，`incremental_update` 会自动检测并直接覆盖。

In [5]:
df_cusip = process_ciq('wrds_cusip', 'cusip', isin_centered=False)
df_cusip = incremental_update(df_cusip, f'{OUTPUT_DIR}/ciq_company_id-cusip.pq', ['ciq_company_id', 'cusip'])
df_cusip.head(3)

-- wrds_cusip <- Z:/dataset/Capital_IQ/helper\wrds_cusip --
  原始: (2707369, 6), cols=['companyid', 'cusip', 'companyname', 'startdate', 'enddate', 'primaryflag']
  清洗后: (2706118, 2)
  首次创建: 2,706,118 rows


,ciq_company_id,cusip,source,update
0,18507,K9682F134,Capital IQ,2026-03-26
1,18511,88579N105,Capital IQ,2026-03-26
2,18511,G4708P104,Capital IQ,2026-03-26


## 4. Capital IQ Company ID <-> Ticker

源表 `wrds_ticker`（字段：companyid, ticker, companyname）。输出 `ciq_company_id-ticker.pq`。

In [6]:
df_ticker = process_ciq('wrds_ticker', 'ticker', isin_centered=False)
df_ticker = incremental_update(df_ticker, f'{OUTPUT_DIR}/ciq_company_id-ticker.pq', ['ciq_company_id', 'ticker'])
df_ticker.head(3)

-- wrds_ticker <- Z:/dataset/Capital_IQ/helper\wrds_ticker --
  原始: (5945321, 6), cols=['companyid', 'ticker', 'companyname', 'startdate', 'enddate', 'primaryflag']
  清洗后: (5835876, 2)
  增量合并: old 4,087,014 + new 5,835,876 -> 5,907,807 (净增 +1,820,793)


,ciq_company_id,ticker,source,update
0,18507,1026283,Capital IQ,2026-03-26
1,18507,1026852,Capital IQ,2026-03-26
2,18507,MMINV,Capital IQ,2026-03-26


## 5. 验证

In [7]:
for name in ['ciq_company_id', 'ciq_company_id-cusip', 'ciq_company_id-ticker']:
    df = pd.read_parquet(f'{OUTPUT_DIR}/{name}.pq')
    print(f'{name}.pq: {df.shape}, cols={list(df.columns)}')
    print(f'  source: {df["source"].unique().tolist()}')
    print(f'  update range: {df["update"].min()} ~ {df["update"].max()}')
    print(f'  sample: {df.iloc[0].to_dict()}')
    print()

ciq_company_id.pq: (18034595, 4), cols=['isin', 'ciq_company_id', 'source', 'update']
  source: ['Capital IQ', 'Sustainalytics']
  update range: 2022-06-17 00:00:00 ~ 2026-03-26 00:00:00
  sample: {'isin': 'AD000A0MYR70', 'ciq_company_id': 216404615, 'source': 'Capital IQ', 'update': Timestamp('2026-03-26 00:00:00')}

ciq_company_id-cusip.pq: (2706118, 4), cols=['ciq_company_id', 'cusip', 'source', 'update']
  source: ['Capital IQ']
  update range: 2026-03-26 00:00:00 ~ 2026-03-26 00:00:00
  sample: {'ciq_company_id': 18507, 'cusip': 'K9682F134', 'source': 'Capital IQ', 'update': Timestamp('2026-03-26 00:00:00')}

ciq_company_id-ticker.pq: (5907807, 4), cols=['ciq_company_id', 'ticker', 'source', 'update']
  source: ['Capital IQ']
  update range: 2022-06-18 00:00:00 ~ 2026-03-26 00:00:00
  sample: {'ciq_company_id': 18507, 'ticker': '1026283', 'source': 'Capital IQ', 'update': Timestamp('2026-03-26 00:00:00')}



## 备注

Capital IQ helper 库还包含以下 WRDS 表，本笔记本未处理（因它们汇入其他聚合表）：

| 源表 | 字段 | 汇入的聚合表 |
|---|---|---|
| `wrds_gvkey` | companyid, gvkey, companyname | `isin/gvkey.pq`（与 Compustat 合并） |
| `wrds_cik` | companyid, cik, companyname | `isin/cik.pq`（纯 Capital IQ，需 join wrds_isin 得 isin <-> cik） |
| `wrds_ciqsymbol` | companyid, 各 symbol | 未单独输出 |
| `ciqgvkeyiid` | gvkey, iid, relatedcompanyid | 未单独输出 |

各表的 `companyname` 字段汇入 `isin/company_name.pq`（与 Refinitiv、Compustat 等合并），详见 `develop.ipynb`。